# **KKBOX Churn Prediction And Retention Intelligence System - Data Preparation and Feature Ingineering**

## **1. Objectives**

## **2. Load Data & Basic Setup**

In [ ]:

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


current_dir = os.getcwd()


while not os.path.exists(os.path.join(current_dir, "data", "raw")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        raise FileNotFoundError("Project root with data/raw not found")
    current_dir = parent


os.chdir(current_dir)

print("Project root:", os.getcwd())
print("data/raw exists:", os.path.exists("data/raw"))


pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

target = "is_churn"

## **3. Light Cleaning**

In [ ]:
# 3.1. Train dataset

train = pd.read_csv("data/raw/train_v2.csv")
train = train.drop_duplicates(subset="msno")

train["is_churn"] = train["is_churn"].astype(int)

In [7]:
# 3.2. Members dataset
members = pd.read_csv("data/raw/members_v3.csv")

members = members.drop_duplicates(subset="msno")

members.loc[(members["bd"] < 0) | (members["bd"] > 100), "bd"] = np.nan

members["gender"] = members["gender"].astype("category")

members["registration_init_time"] = pd.to_datetime(
    members["registration_init_time"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

In [8]:
# 3.3. Transactions dataset
transactions = pd.read_csv("data/raw/transactions_v2.csv")

transactions = transactions.drop_duplicates()

transactions["transaction_date"] = pd.to_datetime(
    transactions["transaction_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

transactions["membership_expire_date"] = pd.to_datetime(
    transactions["membership_expire_date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

num_cols = [
    "payment_method_id", "payment_plan_days", "plan_list_price",
    "actual_amount_paid", "is_auto_renew", "is_cancel"
]

transactions[num_cols] = transactions[num_cols].apply(pd.to_numeric, errors="coerce")


In [10]:
# 3.4. User Logs dataset
transactions_train = transactions.merge(train, on="msno", how="left")

user_logs = pd.read_csv("data/raw/user_logs_v2.csv")

log_cols = [
    "num_25", "num_50", "num_75", "num_985",
    "num_100", "num_unq", "total_secs"
]

user_logs[log_cols] = user_logs[log_cols].apply(pd.to_numeric, errors="coerce")

user_logs["date"] = pd.to_datetime(
    user_logs["date"].astype(str),
    format="%Y%m%d",
    errors="coerce"
)

In [11]:
# 3.5. Quick Sanity Checks

print("Train:", train.shape)
print("Members:", members.shape)
print("Transactions:", transactions.shape)
print("User logs:", user_logs.shape)

print("Missing values (members):")
print(members.isnull().mean().sort_values(ascending=False).head())

Train: (970960, 2)
Members: (6769473, 6)
Transactions: (1431009, 9)
User logs: (18396362, 9)
Missing values (members):
gender            0.654335
bd                0.000835
city              0.000000
msno              0.000000
registered_via    0.000000
dtype: float64
